# Session 2 — Analysis: Sequence Length Scaling

## Background

Attention's memory footprint grows as O(seq²): the attention matrix is `[batch, heads, seq_len, seq_len]` and must be materialized and transposed at every step. Session 1 showed that the GPU's memory bus (~300 GB/s) becomes the bottleneck as batch size grows. Sequence length applies the same pressure through a different dimension, but with a quadratic rather than linear relationship. The TPU's HBM2 bus (~819 GB/s) provides 2.7× more bandwidth — if this is the binding constraint, the gap should widen as sequences grow.

## Goal

At the end of this session, participants will be able to explain how attention's quadratic memory footprint interacts differently with each device's memory architecture, will have measured how the throughput ratio shifts as sequences lengthen, and will know which device to reach for when their research involves long-context models.

## Question

Does the TPU's memory bandwidth advantage compound as sequences grow, or does the crossover point move?

---

**Prerequisites:** Run `01-seq-sweep-gpu.ipynb` and `02-seq-sweep-tpu.ipynb` first to populate `results/gpu_seq_sweep.json` and `results/tpu_seq_sweep.json`.

In [1]:
import json, os

def load_seq_results(results_dir="results"):
    out = {}
    for fname in ["gpu_seq_sweep.json", "tpu_seq_sweep.json"]:
        path = os.path.join(results_dir, fname)
        if not os.path.exists(path):
            print(f"Missing: {path} — run the corresponding benchmark notebook first.")
            continue
        with open(path) as f:
            data = json.load(f)
        hw = data["hardware"]
        out[hw] = {
            "device_name":      data.get("device_name", ""),
            "fixed_batch_size": data.get("fixed_batch_size"),
            "results":          {int(k): v for k, v in data["results"].items()},
        }
    return out

data = load_seq_results("results")

GPU_RESULTS = data.get("GPU", {}).get("results", {})
TPU_RESULTS = data.get("TPU", {}).get("results", {})

print("Loaded:")
for hw, d in data.items():
    print(f"  {hw} ({d['device_name']}) — {len(d['results'])} seq_len points, fixed batch={d['fixed_batch_size']}")

Loaded:
  GPU (NVIDIA L4) — 6 seq_len points, fixed batch=32
  TPU (v5litepod-1) — 6 seq_len points, fixed batch=32


In [2]:
from itertools import chain

hw_order  = [hw for hw in ["GPU", "TPU"] if data.get(hw)]
all_seqs  = sorted(set(chain.from_iterable(data[hw]["results"] for hw in hw_order)))
col       = 22

header = f"{'seq_len':>8}" + "".join(f" | {hw + ' (samples/s)':>{col}}" for hw in hw_order)
if len(hw_order) == 2:
    header += f" | {'TPU/GPU ratio':>{col}}"
print(header)
print("-" * len(header))

for sl in all_seqs:
    row = f"{sl:>8}"
    g = GPU_RESULTS.get(sl)
    t = TPU_RESULTS.get(sl)
    for hw in hw_order:
        v = data[hw]["results"].get(sl)
        label = f"{v:,.0f}" if v else "OOM"
        row += f" | {label:>{col}}"
    if len(hw_order) == 2:
        if g and t:
            row += f" | {t/g:>{col}.2f}"
        else:
            row += f" | {'—':>{col}}"
    print(row)

 seq_len |        GPU (samples/s) |        TPU (samples/s) |          TPU/GPU ratio
-----------------------------------------------------------------------------------
      64 |                  5,895 |                  3,146 |                   0.53
     128 |                  2,906 |                  3,166 |                   1.09
     256 |                  1,249 |                  3,160 |                   2.53
     512 |                    516 |                  3,129 |                   6.06
    1024 |                    196 |                  3,140 |                  16.01
    2048 |                     66 |                  3,128 |                  47.17


In [3]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os

COLORS  = {"GPU": "#76b900", "TPU": "#4285F4"}
MARKERS = {"GPU": "o",       "TPU": "^"}

fig, ax = plt.subplots(figsize=(9, 5))

for hw in hw_order:
    xs = sorted(data[hw]["results"])
    ys = [data[hw]["results"][x] for x in xs]
    ax.plot(xs, ys, marker=MARKERS[hw], linewidth=2, markersize=7,
            color=COLORS[hw], label=hw)

ax.set_xscale("log", base=2)
ax.xaxis.set_major_formatter(mticker.ScalarFormatter())
ax.set_xlabel("Sequence length (tokens)", fontsize=12)
ax.set_ylabel("Throughput (samples / second)", fontsize=12)
ax.set_title("Throughput vs Sequence Length — batch=32, d_model=512", fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, which="both", linestyle="--", alpha=0.4)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout()

os.makedirs("../docs/assets", exist_ok=True)
plt.savefig("../docs/assets/session_2_chart_throughput.png", dpi=150)
print("Saved session_2_chart_throughput.png")

Saved session_2_chart_throughput.png


In [4]:
if "GPU" not in hw_order or "TPU" not in hw_order:
    print("Need both GPU and TPU results for the ratio chart.")
else:
    common = sorted(set(GPU_RESULTS) & set(TPU_RESULTS))
    ratios = [TPU_RESULTS[sl] / GPU_RESULTS[sl] for sl in common]

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(common, ratios, marker="D", linewidth=2, markersize=7, color="#e67e22")
    ax.axhline(1.0, color="gray", linestyle="--", linewidth=1, label="parity (ratio = 1)")

    ax.set_xscale("log", base=2)
    ax.xaxis.set_major_formatter(mticker.ScalarFormatter())
    ax.set_xlabel("Sequence length (tokens)", fontsize=12)
    ax.set_ylabel("TPU / GPU throughput ratio", fontsize=12)
    ax.set_title("TPU Advantage vs Sequence Length (ratio > 1 → TPU faster)", fontsize=13)
    ax.legend(fontsize=11)
    ax.grid(True, which="both", linestyle="--", alpha=0.4)
    plt.tight_layout()

    plt.savefig("../docs/assets/session_2_chart_ratio.png", dpi=150)
    print("Saved session_2_chart_ratio.png")

    print(f"\nTPU/GPU ratio at each seq_len:")
    for sl, r in zip(common, ratios):
        print(f"  seq_len={sl:>5} → {r:.2f}×")

Saved session_2_chart_ratio.png

TPU/GPU ratio at each seq_len:
  seq_len=   64 → 0.53×
  seq_len=  128 → 1.09×
  seq_len=  256 → 2.53×
  seq_len=  512 → 6.06×
  seq_len= 1024 → 16.01×
  seq_len= 2048 → 47.17×


In [5]:
# Chart: Tokens per second = throughput (samples/sec) × seq_len
# GPU processes FEWER tokens/sec as sequences lengthen (quadratic memory pressure).
# TPU processes MORE tokens/sec — bandwidth absorbs the larger O(seq²) traffic.
if "GPU" not in hw_order or "TPU" not in hw_order:
    print("Need both GPU and TPU results.")
else:
    fig, ax = plt.subplots(figsize=(9, 5))

    for hw in hw_order:
        xs  = sorted(data[hw]["results"])
        ys  = [data[hw]["results"][x] * x for x in xs]   # tokens/sec = samples/sec × seq_len
        ax.plot(xs, ys, marker=MARKERS[hw], linewidth=2, markersize=7,
                color=COLORS[hw], label=hw)

    ax.set_xscale("log", base=2)
    ax.set_yscale("log")
    ax.xaxis.set_major_formatter(mticker.ScalarFormatter())
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(
        lambda x, _: f"{x/1e6:.1f}M" if x >= 1e6 else f"{x/1e3:.0f}k"
    ))
    ax.set_xlabel("Sequence length (tokens)", fontsize=12)
    ax.set_ylabel("Token throughput (tokens / second)", fontsize=12)
    ax.set_title("Token Throughput vs Seq Length — GPU collapses, TPU scales", fontsize=13)
    ax.legend(fontsize=11)
    ax.grid(True, which="both", linestyle="--", alpha=0.4)
    plt.tight_layout()

    plt.savefig("../docs/assets/session_2_chart_tokens.png", dpi=150)
    print("Saved session_2_chart_tokens.png")

    print("\nToken throughput (samples/sec × seq_len):")
    all_seqs = sorted(set(data["GPU"]["results"]) | set(data["TPU"]["results"]))
    print(f"  {'seq_len':>8}  {'GPU tokens/s':>16}  {'TPU tokens/s':>16}  {'TPU/GPU':>8}")
    print(f"  {'-'*8}  {'-'*16}  {'-'*16}  {'-'*8}")
    for sl in all_seqs:
        g = data["GPU"]["results"].get(sl)
        t = data["TPU"]["results"].get(sl)
        g_tok = g * sl if g else None
        t_tok = t * sl if t else None
        ratio_str = f"{t_tok/g_tok:.1f}×" if g_tok and t_tok else "—"
        g_str = f"{g_tok:,.0f}" if g_tok else "OOM"
        t_str = f"{t_tok:,.0f}" if t_tok else "OOM"
        print(f"  {sl:>8}  {g_str:>16}  {t_str:>16}  {ratio_str:>8}")

Saved session_2_chart_tokens.png

Token throughput (samples/sec × seq_len):
   seq_len      GPU tokens/s      TPU tokens/s   TPU/GPU
  --------  ----------------  ----------------  --------
        64           377,293           201,350      0.5×
       128           371,968           405,299      1.1×
       256           319,667           808,960      2.5×
       512           264,294         1,601,997      6.1×
      1024           200,909         3,215,770     16.0×
      2048           135,782         6,405,325     47.2×


## Interpreting the results

| Observation | What it means |
|---|---|
| GPU throughput declines as seq_len grows | Attention's O(seq²) matrix increasingly saturates the 300 GB/s memory bus |
| TPU maintains or extends its scaling | 819 GB/s HBM2 bandwidth absorbs the growing memory traffic longer |
| TPU/GPU ratio grows with seq_len | The same bandwidth gap that drove Session 1 results compounds quadratically |
| Either device hits OOM at large seq_len | Memory, not compute, becomes the binding constraint at long contexts |

## Connection to Session 1

Session 1 showed the crossover at `batch=32` with `seq_len=128` fixed. Session 2 holds
`batch=32` and moves `seq_len`. The TPU's bandwidth advantage — the same root cause as
Session 1 — now compounds *quadratically* rather than linearly, because every doubling of
sequence length quadruples the attention matrix.

## Decision guidance

If your model operates on long sequences (≥ 512 tokens) at moderate batch sizes, the TPU's
bandwidth advantage is stronger than the batch-size crossover chart from Session 1 alone
suggests. Conversely, if you are experimenting with very small batches at short sequences,
the GPU remains competitive and avoids XLA compilation overhead (see Session 7).